In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Optional safety setting in case Spark is strict on malformed casts
spark.conf.set("spark.sql.ansi.enabled", "false")

# =========================
# 1. Read Bronze tables
# =========================
patient_bronze = spark.table("bronze_patient_master")
provider_bronze = spark.table("bronze_provider_roster")
dept_bronze = spark.table("bronze_department_reference")
visit_bronze = spark.table("bronze_er_event_log")
followup_bronze = spark.table("bronze_followup_log")

In [0]:
# =========================
# 2. Patient Silver
# =========================
patient_silver = (
    patient_bronze
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("gender_cd", F.upper(F.trim(F.col("gender_cd"))))
    .withColumn("age_band_raw", F.trim(F.col("age_band_raw")))
    .withColumn("insurance_plan", F.initcap(F.trim(F.col("insurance_plan"))))
    .withColumn("lang_pref", F.initcap(F.trim(F.col("lang_pref"))))
    .withColumn("risk_ind", F.upper(F.trim(F.col("risk_ind"))))
    .withColumn("zip3", F.trim(F.col("zip3")))
    .withColumn("chronic_cnt", F.col("chronic_cnt").cast("int"))
    .filter(F.col("patient_id").isNotNull())
    .dropDuplicates(["patient_id"])
)

patient_silver.write.mode("overwrite").format("delta").saveAsTable("silver_patient")


In [0]:
# =========================
# 3. Provider Silver
# =========================
provider_silver = (
    provider_bronze
    .withColumn("provider_id", F.trim(F.col("provider_id")))
    .withColumn("provider_role_raw", F.initcap(F.trim(F.col("provider_role_raw"))))
    .withColumn("shift_code", F.upper(F.trim(F.col("shift_code"))))
    .withColumn("active_ind", F.upper(F.trim(F.col("active_ind"))))
    .withColumn("exp_yrs", F.col("exp_yrs").cast("int"))
    .filter(F.col("provider_id").isNotNull())
    .dropDuplicates(["provider_id"])
)

provider_silver.write.mode("overwrite").format("delta").saveAsTable("silver_provider")

In [0]:
# =========================
# 4. Department Silver
# =========================
dept_silver = (
    dept_bronze
    .withColumn("dept_code", F.upper(F.trim(F.col("dept_code"))))
    .withColumn("dept_name", F.initcap(F.trim(F.col("dept_name_raw"))))
    .withColumn("hospital_name", F.initcap(F.trim(F.col("hospital_name"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
    .filter(F.col("dept_code").isNotNull())
    .dropDuplicates(["dept_code"])
)

dept_silver.write.mode("overwrite").format("delta").saveAsTable("silver_department")

In [0]:
# =========================
# 5. ER Visit Silver
# =========================
visit_clean = (
    visit_bronze
    .withColumn("visit_id", F.trim(F.col("visit_id")))
    .withColumn("patient_id", F.trim(F.col("patient_id")))
    .withColumn("provider_id", F.trim(F.col("provider_id")))
    .withColumn("dept_code", F.upper(F.trim(F.col("dept_code"))))
    .withColumn("arrival_mode_cd", F.upper(F.trim(F.col("arrival_mode_cd"))))
    .withColumn("acuity_cd", F.col("acuity_cd").cast("int"))
    .withColumn("chief_complaint_raw", F.initcap(F.trim(F.col("chief_complaint_raw"))))
    .withColumn("dispo_cd", F.upper(F.trim(F.col("dispo_cd"))))
    .withColumn("visit_status_cd", F.upper(F.trim(F.col("visit_status_cd"))))
    .withColumn("src_system_nm", F.trim(F.col("src_system_nm")))
    .withColumn("load_batch_id", F.trim(F.col("load_batch_id")))
)

# Safe timestamp parsing for mixed raw formats
visit_clean = (
    visit_clean
    .withColumn(
        "arrival_ts_parsed",
        F.coalesce(
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("arrival_ts", "MM/dd/yyyy HH:mm"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("arrival_ts", "yyyy-MM-dd HH:mm:ss"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("arrival_ts", "yyyy-MM-dd HH:mm")))
        )
    )
    .withColumn(
        "triage_ts_parsed",
        F.coalesce(
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("triage_ts", "MM/dd/yyyy HH:mm"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("triage_ts", "yyyy-MM-dd HH:mm:ss"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("triage_ts", "yyyy-MM-dd HH:mm")))
        )
    )
    .withColumn(
        "bed_ts_parsed",
        F.coalesce(
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("bed_ts", "MM/dd/yyyy HH:mm"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("bed_ts", "yyyy-MM-dd HH:mm:ss"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("bed_ts", "yyyy-MM-dd HH:mm")))
        )
    )
    .withColumn(
        "md_seen_ts_parsed",
        F.coalesce(
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("md_seen_ts", "MM/dd/yyyy HH:mm"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("md_seen_ts", "yyyy-MM-dd HH:mm:ss"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("md_seen_ts", "yyyy-MM-dd HH:mm")))
        )
    )
    .withColumn(
        "discharge_ts_parsed",
        F.coalesce(
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("discharge_ts", "MM/dd/yyyy HH:mm"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("discharge_ts", "yyyy-MM-dd HH:mm:ss"))),
            F.to_timestamp(F.from_unixtime(F.unix_timestamp("discharge_ts", "yyyy-MM-dd HH:mm")))
        )
    )
    .drop("arrival_ts", "triage_ts", "bed_ts", "md_seen_ts", "discharge_ts")
    .withColumnRenamed("arrival_ts_parsed", "arrival_ts")
    .withColumnRenamed("triage_ts_parsed", "triage_ts")
    .withColumnRenamed("bed_ts_parsed", "bed_ts")
    .withColumnRenamed("md_seen_ts_parsed", "md_seen_ts")
    .withColumnRenamed("discharge_ts_parsed", "discharge_ts")
)


In [0]:
# Deduplicate on visit_id using latest ingestion timestamp
w_visit = Window.partitionBy("visit_id").orderBy(F.col("ingestion_ts").desc())

visit_silver = (
    visit_clean
    .withColumn("rn", F.row_number().over(w_visit))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [0]:
# Basic data quality filters
visit_silver = (
    visit_silver
    .filter(F.col("visit_id").isNotNull())
    .filter(F.col("patient_id").isNotNull())
    .filter(F.col("arrival_ts").isNotNull())
    .filter(F.col("acuity_cd").between(1, 5))
)

# Standardize code columns into readable labels
visit_silver = (
    visit_silver
    .withColumn(
        "arrival_mode",
        F.when(F.col("arrival_mode_cd").isin("AMB", "AMBULANCE"), "Ambulance")
         .when(F.col("arrival_mode_cd").isin("WALK", "WALKIN", "WALK IN"), "Walk In")
         .when(F.col("arrival_mode_cd").isin("TRANS", "TRANSFER"), "Transfer")
         .otherwise("Other")
    )
    .withColumn(
        "disposition",
        F.when(F.col("dispo_cd").isin("DISCH", "DISCHARGED"), "Discharged")
         .when(F.col("dispo_cd").isin("ADMIT", "ADMITTED"), "Admitted")
         .when(F.col("dispo_cd").isin("OBS", "OBSERVATION"), "Observation")
         .when(F.col("dispo_cd").isin("TRANS", "TRANSFERRED"), "Transferred")
         .when(F.col("dispo_cd").isin("AMA"), "AMA")
         .when(F.col("dispo_cd").isin("EXPIRED"), "Expired")
         .otherwise("Other")
    )
    .withColumn(
        "visit_status",
        F.when(F.col("visit_status_cd").isin("COMP", "COMPLETED"), "Completed")
         .when(F.col("visit_status_cd").isin("LWBS"), "LWBS")
         .when(F.col("visit_status_cd").isin("CANC", "CANCELLED"), "Cancelled")
         .otherwise("Other")
    )
)


In [0]:
# Standardize integer indicator fields to 0 or 1
visit_indicator_cols = [
    "lab_ord_ind", "img_ord_ind", "med_ord_ind", "consult_ord_ind",
    "admit_ind", "obs_ind", "lwbs_ind", "revisit_30d_ind"
]

for c in visit_indicator_cols:
    visit_silver = visit_silver.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit(0))
         .when(F.col(c) >= 1, F.lit(1))
         .otherwise(F.lit(0))
    )

visit_silver.write.mode("overwrite").format("delta").saveAsTable("silver_er_visit")

In [0]:
# =========================
# 6. Followup Silver
# =========================
followup_clean = (
    followup_bronze
    .withColumn("followup_id", F.trim(F.col("followup_id")))
    .withColumn("visit_id", F.trim(F.col("visit_id")))
    .withColumn("followup_type_cd", F.upper(F.trim(F.col("followup_type_cd"))))
    .withColumn("rec_followup_days", F.col("rec_followup_days").cast("int"))
    .withColumn("days_to_followup", F.col("days_to_followup").cast("double"))
    .withColumn("src_system_nm", F.trim(F.col("src_system_nm")))
)

followup_silver = (
    followup_clean
    .withColumn(
        "followup_type",
        F.when(F.col("followup_type_cd").isin("PCP"), "PCP")
         .when(F.col("followup_type_cd").isin("SPEC", "SPECIALIST"), "Specialist")
         .when(F.col("followup_type_cd").isin("IMG", "IMAGING"), "Imaging")
         .when(F.col("followup_type_cd").isin("LAB"), "Lab")
         .when(F.col("followup_type_cd").isin("CARD"), "Cardiology")
         .when(F.col("followup_type_cd").isin("ORTHO"), "Orthopedics")
         .when(F.col("followup_type_cd").isin("BH", "BEHAVIORAL"), "Behavioral Health")
         .otherwise("Other")
    )
)

followup_indicator_cols = [
    "followup_needed_ind",
    "scheduled_ind",
    "completed_ind",
    "completed_target_ind"
]

for c in followup_indicator_cols:
    followup_silver = followup_silver.withColumn(
        c,
        F.when(F.col(c).isNull(), F.lit(0))
         .when(F.col(c) >= 1, F.lit(1))
         .otherwise(F.lit(0))
    )

w_follow = Window.partitionBy("followup_id").orderBy(F.col("ingestion_ts").desc())

followup_silver = (
    followup_silver
    .withColumn("rn", F.row_number().over(w_follow))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

followup_silver.write.mode("overwrite").format("delta").saveAsTable("silver_followup")

In [0]:
# =========================
# 7. Validation checks
# =========================
print("silver_patient")
spark.sql("SELECT COUNT(*) AS row_count FROM silver_patient").show()

print("silver_provider")
spark.sql("SELECT COUNT(*) AS row_count FROM silver_provider").show()

print("silver_department")
spark.sql("SELECT COUNT(*) AS row_count FROM silver_department").show()

print("silver_er_visit")
spark.sql("SELECT COUNT(*) AS row_count FROM silver_er_visit").show()

print("silver_followup")
spark.sql("SELECT COUNT(*) AS row_count FROM silver_followup").show()

print("visit duplicate check")
spark.sql("""
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT visit_id) AS distinct_visits
FROM silver_er_visit
""").show()

print("parsed timestamp null counts")
spark.sql("""
SELECT
    SUM(CASE WHEN arrival_ts IS NULL THEN 1 ELSE 0 END) AS arrival_ts_nulls,
    SUM(CASE WHEN triage_ts IS NULL THEN 1 ELSE 0 END) AS triage_ts_nulls,
    SUM(CASE WHEN bed_ts IS NULL THEN 1 ELSE 0 END) AS bed_ts_nulls,
    SUM(CASE WHEN md_seen_ts IS NULL THEN 1 ELSE 0 END) AS md_seen_ts_nulls,
    SUM(CASE WHEN discharge_ts IS NULL THEN 1 ELSE 0 END) AS discharge_ts_nulls
FROM silver_er_visit
""").show()

print("sample visit rows")
spark.sql("""
SELECT
    visit_id,
    patient_id,
    provider_id,
    dept_code,
    arrival_ts,
    triage_ts,
    bed_ts,
    md_seen_ts,
    discharge_ts,
    arrival_mode,
    acuity_cd,
    chief_complaint_raw,
    disposition,
    visit_status
FROM silver_er_visit
LIMIT 10
""").show(truncate=False)

silver_patient
+---------+
|row_count|
+---------+
|    16500|
+---------+

silver_provider
+---------+
|row_count|
+---------+
|       58|
+---------+

silver_department
+---------+
|row_count|
+---------+
|        1|
+---------+

silver_er_visit
+---------+
|row_count|
+---------+
|    20966|
+---------+

silver_followup
+---------+
|row_count|
+---------+
|     9336|
+---------+

visit duplicate check
+----------+---------------+
|total_rows|distinct_visits|
+----------+---------------+
|     20966|          20966|
+----------+---------------+

parsed timestamp null counts
+----------------+---------------+------------+----------------+------------------+
|arrival_ts_nulls|triage_ts_nulls|bed_ts_nulls|md_seen_ts_nulls|discharge_ts_nulls|
+----------------+---------------+------------+----------------+------------------+
|               0|           6969|        6957|            6995|              6957|
+----------------+---------------+------------+----------------+-----------------